# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
[1. 2. 3.] [0 1 1]
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
[4. 4. 4.]
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
[[0 1 2][1 2 3][2 3 4]]
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
6.0
#5
print((np.arange(1, 5) ** 2).mean())
7.5
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [9]:
# 1
sept=amount[clean & (month == "2026-09")].sum()
octo=amount[clean & (month == "2026-10")].sum()
change=(sept/octo-1)*100

In [10]:
sept

np.float64(35243410.0)

In [11]:
octo

np.float64(34419520.0)

In [12]:
change 

np.float64(2.3936708007549212)

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [22]:
# 2
categories = np.unique(cat)
months = np.unique(month)
table_cat = np.array([
    [ amount[clean & (cat == c) & (month == m)].sum()
        for m in months]
    for c in categories])
total_cat = table_cat.sum(axis=1)
best_idx = np.argmax(total_cat)
change_cat=(table_cat[:, 1] / table_cat[:, 0] - 1) * 100
worst_idx = np.argmin(change_cat)

In [23]:
print(categories)
print(months)
print(table_cat)

['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
['2026-09' '2026-10']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]


In [24]:
print(total_cat)
print(categories[best_idx])
print(total_cat[best_idx])

[10123520.  2206210.  5289940. 17672830. 34370430.]
Электроника
34370430.0


In [27]:
print(np.round(change_cat, 2))
print(categories[worst_idx])
print(round(change_cat[worst_idx], 2))

[ 2.56  7.1   2.61 -7.21 -2.5 ]
Одежда
-7.21


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [29]:
# 3
count_table = np.array([
    [
        np.sum(clean & (cat == c) & (month == m))
        for m in months
    ]
    for c in categories
])

print("Number of orders:")
print(count_table)

Number of orders:
[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]


In [31]:
avg_check= table_cat / count_table
oct_avg = avg_check[:, 1]
best_idx = np.argmax(oct_avg)

In [32]:
print(categories[best_idx])
print(round(oct_avg[best_idx],2))

Электроника
33600.2


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [37]:
cities = np.unique(city)
months = np.unique(month)

table = np.array([
    [
        amount[clean & (city == c) & (month == m)].sum()
        for m in months
    ]
    for c in cities
])
share = table / table.sum(axis=0)

In [38]:
print(np.round(share * 100, 2))

[[ 7.31  7.38]
 [41.15 42.54]
 [25.63 25.71]
 [ 2.62  2.82]
 [10.35  9.8 ]
 [12.93 11.75]]


In [39]:
print(share.sum(axis=0))

[1. 1.]


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [42]:
# 5
change_city = table[:, 1] - table[:, 0]
total_change = change_city.sum()
contribution = change_city / total_change * 100
shymkent_idx = np.where(cities == "Шымкент")[0][0]

In [41]:
print(change_city)
print(total_change)

[ -36540.  141950. -183690.   45870. -276840. -514640.]
-823890.0


In [43]:
print(np.round(contribution,2))
print(contribution[shymkent_idx],2)

[  4.44 -17.23  22.3   -5.57  33.6   62.46]
62.46464940708104 2


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [44]:
# 6
share = table / table.sum(axis=0)

print(np.round(share * 100, 2))

[[ 7.31  7.38]
 [41.15 42.54]
 [25.63 25.71]
 [ 2.62  2.82]
 [10.35  9.8 ]
 [12.93 11.75]]


In [45]:
print(share.sum(axis=0))

[1. 1.]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [46]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
print(z.mean())
print(z.std())

7.04695264417065e-17
1.0


In [47]:
outliers = np.abs(z) > 3
count = outliers.sum()
percentage = count / len(z) * 100
print(count)
print(round(percentage,2))

87
1.84


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8